# Filter training
In this notebook we train the filter with the labeled data produced in the previous notebook.  First, we embed the concatenated title+description via `FinBERT`, and then we cross-validate the following models:
- a simple logistic regression (baseline),
- a cosine-kNN model,
- a shallow neural network with 1 hidden layers with 8/16/32 nodes.

### Data loading and train-test split
We first load the labeled data

In [1]:
import pandas as pd

df = pd.read_csv("data/labeled_headlines.csv", index_col=False)
df = df[df.relevant != -1]

Then we embed each sentence

In [2]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("ProsusAI/finbert")
df["embeddings"] = embedder.encode(df.concat.to_list(), show_progress_bar=True).tolist()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: ProsusAI/finbert
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/50 [00:00<?, ?it/s]

and produce a train-test split:

In [3]:
from sklearn.model_selection import train_test_split
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(
    df.embeddings, df.relevant, shuffle=True
)

# dataframe versions for sklearn
X_train_df = pd.DataFrame(X_train.to_list())
X_test_df = pd.DataFrame(X_test.to_list())

# numpy version for keras
X_train_np = np.stack(X_train.values)
X_test_np = np.stack(X_test.values)
y_train_np = y_train.values
y_test_np = y_test.values

### Models definitions
##### Basic logistic regression (baseline)

In [4]:
from sklearn.linear_model import LogisticRegressionCV

logit = LogisticRegressionCV(cv=5)

##### $k$-nearest neighbors

In [5]:
from sklearn.neighbors import KNeighborsClassifier

neigh = KNeighborsClassifier(n_neighbors=5, metric="cosine", weights="distance")

##### Shallow neural networks

In [6]:
import os

os.environ["KERAS_BACKEND"] = "torch"

import keras


def generate_shallow(nodes: int, summarize: bool = True) -> keras.Model:
    inputs = keras.Input(shape=(embedder.config.hidden_size,))
    x = keras.layers.Dense(nodes, activation="relu")(inputs)
    x = keras.layers.Dropout(0.5)(x)
    outputs = keras.layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(
        inputs=inputs, outputs=outputs, name=f"relevance_filter ({nodes} nodes)"
    )
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

    if summarize:
        model.summary()

    return model

In [7]:
nn8 = generate_shallow(8)
nn16 = generate_shallow(16)
nn32 = generate_shallow(32)
nn64 = generate_shallow(64)

Model: "relevance_filter (8 nodes)"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8)              │         6,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,161 (24.07 KB)

 Trainable params: 6,161 (24.07 KB)

 Non-trainable params: 0 (0.00 B)

...

Model: "relevance_filter (64 nodes)"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │        49,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 49,281 (192.50 KB)

 Trainable params: 49,281 (192.50 KB)

 Non-trainable params: 0 (0.00 B)

### Model selection

In [8]:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score

callback = keras.callbacks.EarlyStopping(patience=2)

kf = KFold(n_splits=5, shuffle=True)

results = []

for train_index, holdout_index in kf.split(X_train):
    # pandas
    X_tt_df = X_train_df.iloc[train_index]
    y_tt_df = y_train.iloc[train_index]
    X_th_df = X_train_df.iloc[holdout_index]
    y_th_df = y_train.iloc[holdout_index]

    # numpy
    X_tt_np = X_train_np[train_index]
    X_th_np = X_train_np[holdout_index]
    y_tt_np = y_train_np[train_index]
    y_th_np = y_train_np[holdout_index]

    scores = {}

    # logistic regression
    logit.fit(X_tt_df, y_tt_df)
    scores["logit"] = accuracy_score(y_th_df, logit.predict(X_th_df))

    # kNN
    neigh.fit(X_tt_df, y_tt_df)
    scores["kNN"] = accuracy_score(y_th_df, neigh.predict(X_th_df))

    # nn8
    nn8.fit(X_tt_np, y_tt_np, batch_size=64, epochs=15, callbacks=[callback], verbose=0)
    scores["nn8"] = accuracy_score(y_th_np, np.rint(nn8.predict(X_th_np)))

    # nn16
    nn16.fit(
        X_tt_np, y_tt_np, batch_size=64, epochs=15, callbacks=[callback], verbose=0
    )
    scores["nn16"] = accuracy_score(y_th_np, np.rint(nn16.predict(X_th_np, verbose=0)))

    # nn32
    nn32.fit(
        X_tt_np, y_tt_np, batch_size=64, epochs=15, callbacks=[callback], verbose=0
    )
    scores["nn32"] = accuracy_score(y_th_np, np.rint(nn32.predict(X_th_np, verbose=0)))

    # nn64
    nn64.fit(
        X_tt_np, y_tt_np, batch_size=64, epochs=15, callbacks=[callback], verbose=0
    )
    scores["nn64"] = accuracy_score(y_th_np, np.rint(nn64.predict(X_th_np, verbose=0)))

    results.append(scores)

results = pd.DataFrame(results)

/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(
/Users/

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)
/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)
/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)
/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)
/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)
/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)
/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,

...

/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)
/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)
/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)
/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)
/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)
/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)
/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)
/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)
/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)


In [9]:
results

,logit,kNN,nn8,nn16,nn32,nn64
0,0.962343,0.962343,0.958159,0.966527,0.970711,0.966527
1,0.974790,0.941176,0.978992,0.987395,0.987395,0.978992
2,0.957983,0.928571,0.966387,0.970588,0.978992,0.983193
3,0.941176,0.936975,0.966387,0.974790,0.983193,0.987395
4,0.945378,0.915966,0.974790,0.983193,0.991597,0.995798


In [10]:
summary = pd.concat([results.mean(), results.std()], keys=["mean", "std"], axis=1)
summary

,mean,std
logit,0.956334,0.013502
kNN,0.937006,0.017126
nn8,0.968943,0.008132
nn16,0.976499,0.008672
nn32,0.982378,0.008037
nn64,0.982381,0.010824


The logistic regression is already a great baseline. However, it might be worth considering also `nn16` and `nn32`.

In [11]:
# logit
logit.fit(X_train_df, y_train)

# nn16
nn16.fit(
    X_train_np, y_train_np, batch_size=64, epochs=15, callbacks=[callback], verbose=0
)

# summary
print()
print("Logit accuracy score:", accuracy_score(y_test, logit.predict(X_test_df)))
print(
    "nn16 accuracy score: ",
    accuracy_score(y_test_np, np.rint(nn16.predict(X_test_np, verbose=0))),
)

/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/Users/ralbesia/Projects/jitter/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(
/Users/


Logit accuracy score: 0.9195979899497487
nn16 accuracy score:  0.9221105527638191


Since `nn16` still performs better than the baseline, and since it allows greater flexibility later on when we will also incorporate user feedback, it will be our model of choice.